<center>

**Text Mining Project** <br> Master in Data Science and Advanced Analytics <br>

NOVA Information Management School <br> Universidade Nova de Lisboa

</center>

<center> 
<b><font size="6" color="#0ee071">Tweets-Based Market Sentiment</font></b> 
</center>


<font color="#0ee071"> **Group 25 - June 2025** </font>

- Beatris Daicu, 20221854
- Diogo Carvalho, 20221935
- Ricardo Pereira, 20250343
- Yehor Malakhov, 20221691


In [1]:
from typing import TYPE_CHECKING, cast

import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizer,
)

if TYPE_CHECKING:
    import numpy as np
    from transformers.modeling_outputs import SequenceClassifierOutput

In [2]:
MAX_LEN = 64
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

In [3]:
test = pd.read_csv("data/test.csv", index_col=0)

In [4]:
tokenizer: PreTrainedTokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased"
)
model: PreTrainedModel = AutoModelForSequenceClassification.from_pretrained(
    "results/transformer_outputs_bert-base-multilingual-cased", num_labels=3
)
model.to(DEVICE)  # pyright: ignore[reportArgumentType]
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,)

In [5]:
inputs = tokenizer(
    test["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="pt",
)

inputs_dict: dict[str, torch.Tensor] = {
    k: v.to(DEVICE) for k, v in inputs.items()
}

with torch.no_grad():
    outputs: SequenceClassifierOutput = model(**inputs_dict)
    preds: np.ndarray[tuple[int], np.dtype[np.int64]] = (
        cast("torch.Tensor", outputs.logits).argmax(dim=-1).cpu().numpy()
    )

c:\Users\carva\Documents\GitHub\TM_yehor\.venv\Lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at B:\src\torch\aten\src\ATen\native\transformers\hip\sdp_utils.cpp:384.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [8]:
test["label"] = preds
test.to_csv("results/pred_25.csv", columns=["label"])